# Yelp 리뷰 프롬프트 기반 증강 (Type 3 Rewrite)

전처리된 시드(`sampled_df.parquet`, 10,000행)를 두 LLM으로 각각 5,000건씩 재작성하여 인간 작성 리뷰와 AI 재작성 리뷰가 섞인 데이터셋을 만든다.

**모델**: `meta-llama/Llama-3.2-1B-Instruct` (5,000) · `Qwen/Qwen3-1.7B` (5,000)

**산출물**
- `yelp_type3_rewritten_reviews_llama.parquet` / `..._qwen.parquet` — 모델별 생성 결과
- `yelp_type3_rewritten_reviews.parquet` — 두 모델 결합 (10,000행)
- `data_yelp.parquet` — 원본(human) + 재작성(ai) 라벨링 (20,000행)

### 라이브러리 불러오기

In [ ]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

### 경로 설정

In [ ]:
DATA_DIR = Path("../data")

INPUT_PATH = DATA_DIR / "sampled_df.parquet"

LLAMA_OUTPUT_PATH = DATA_DIR / "yelp_type3_rewritten_reviews_llama.parquet"
QWEN_OUTPUT_PATH = DATA_DIR / "yelp_type3_rewritten_reviews_qwen.parquet"
REWRITE_OUTPUT_PATH = DATA_DIR / "yelp_type3_rewritten_reviews.parquet"
LABELED_OUTPUT_PATH = DATA_DIR / "data_yelp.parquet"

### 전처리된 시드 데이터 로드

In [ ]:
sampled_df = pd.read_parquet(INPUT_PATH)
print("Loaded shape:", sampled_df.shape)
print(sampled_df["review_stars"].value_counts().sort_index())
sampled_df.head()

### 모델별 데이터 분할 (각 5,000건)

- 동일 시드(42)로 셔플 후, 앞 5,000건은 Llama / 다음 5,000건은 Qwen에 할당 (중복 없음)

In [ ]:
N_PER_MODEL = 5000
SPLIT_RANDOM_STATE = 42

shuffled = sampled_df.sample(frac=1, random_state=SPLIT_RANDOM_STATE).reset_index(drop=True)

llama_df = shuffled.iloc[:N_PER_MODEL].reset_index(drop=True)
qwen_df = shuffled.iloc[N_PER_MODEL:N_PER_MODEL * 2].reset_index(drop=True)

print("Llama subset:", llama_df.shape)
print("Qwen subset:", qwen_df.shape)

### Rewrite 프롬프트 함수 (Type 3)

- 원본의 감정·평점·핵심 경험은 유지하되, 문장을 그대로 베끼거나 없는 정보를 추가하지 않도록 지시

In [ ]:
def build_prompt_type3_yelp(original_review):
    return f"""
Rewrite the following Yelp review as a new English review written by another customer.

Original review:
{original_review}

Instructions:
Preserve the overall sentiment, rating level, main experience, and key positive or negative aspects.
Do not copy sentences, clauses, or distinctive phrases from the original review.
Do not add unsupported details such as exact dates, employee names, menu items, prices, discounts, or visit times.
Do not mention the business name in the rewritten review.
Do not make the review more extremely positive or negative than the original.
Use natural Yelp-style language appropriate for the business category.
Return only the rewritten review text.
""".strip()

In [ ]:
sample_prompt = build_prompt_type3_yelp(original_review=sampled_df.loc[0, "review_text"])
print(sample_prompt)

### 모델 ID 및 HuggingFace 토큰 설정

- 게이트된 Llama 모델 접근을 위해 환경변수 `HF_TOKEN`을 미리 설정해야 함 (예: `export HF_TOKEN=...`)

In [ ]:
# 게이트 모델(Llama) 접근 토큰: 환경변수에서 읽어옴
# 터미널에서 `export HF_TOKEN=hf_xxx` 또는 아래 줄을 직접 설정
# os.environ["HF_TOKEN"] = "hf_..."

LLAMA_MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
QWEN_MODEL_ID = "Qwen/Qwen3-1.7B"

### 모델 로드 함수

- CUDA → MPS → CPU 순으로 디바이스를 선택하고, pad_token이 없으면 eos_token으로 대체

In [ ]:
def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if torch.cuda.is_available():
        dtype, device = torch.float16, "cuda"
    elif torch.backends.mps.is_available():
        dtype, device = torch.float16, "mps"
    else:
        dtype, device = torch.float32, "cpu"

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=dtype,
        low_cpu_mem_usage=True,
        attn_implementation="eager",
    ).to(device)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model.eval()
    return tokenizer, model

### 생성 설정

- Type 3 rewrite에서는 의미 변질을 막기 위해 temperature를 0.7 수준으로 사용

In [ ]:
GENERATION_CONFIG = {
    "max_new_tokens": 220,
    "temperature": 0.7,
    "top_p": 0.9,
    "do_sample": True,
    "repetition_penalty": 1.05,
}
print(GENERATION_CONFIG)

### 프롬프트 포맷 및 생성 텍스트 후처리 함수

- chat template 지원 시 chat 형식으로 변환, 미지원 시 일반 prompt 사용
- 모델이 붙이는 지시문성 접두어/감싸는 따옴표 제거

In [ ]:
def format_prompt_for_model(prompt, tokenizer):
    messages = [{"role": "user", "content": prompt}]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except Exception:
        return prompt

In [ ]:
def clean_generated_text(text):
    text = str(text).strip()

    prefixes = [
        "Rewritten review:",
        "Review:",
        "Here is the rewritten review:",
        "Here is a rewritten version:",
        "Sure, here is the rewritten review:",
    ]
    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()

    if len(text) >= 2:
        if (text[0] == '"' and text[-1] == '"') or (text[0] == "'" and text[-1] == "'"):
            text = text[1:-1].strip()
    return text

### 단일 프롬프트 생성 함수

In [ ]:
@torch.no_grad()
def generate_rewrite(prompt, tokenizer, model):
    formatted_prompt = format_prompt_for_model(prompt, tokenizer)

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output_ids = model.generate(
        **inputs,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        **GENERATION_CONFIG,
    )

    input_length = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0][input_length:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    del inputs, output_ids, generated_ids
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

    return clean_generated_text(generated_text)

### 생성용 데이터프레임 준비

- 원본 행에 프롬프트와 생성 메타데이터(모델명·하이퍼파라미터·생성시각) 컬럼을 추가

In [ ]:
def prepare_generation_df(subset_df, model_id):
    generation_df = subset_df.copy()
    generation_df["prompt"] = generation_df["review_text"].apply(build_prompt_type3_yelp)
    generation_df["rewritten_review_text"] = None
    generation_df["model_name"] = model_id
    generation_df["temperature"] = GENERATION_CONFIG["temperature"]
    generation_df["top_p"] = GENERATION_CONFIG["top_p"]
    generation_df["max_new_tokens"] = GENERATION_CONFIG["max_new_tokens"]
    generation_df["created_at"] = datetime.now().isoformat(timespec="seconds")
    return generation_df

### 전체 생성 루프 (중간 저장 포함)

- 생성 도중 오류가 나도 결과를 보존하기 위해 `SAVE_EVERY`마다 parquet로 체크포인트 저장
- ⚠️ 10,000건 생성은 시간이 오래 걸린다 (GPU 권장). RunPod 등 GPU 환경에서 실행 권장.

In [ ]:
SAVE_EVERY = 50


def run_generation(subset_df, model_id, output_path):
    tokenizer, model = load_model(model_id)
    generation_df = prepare_generation_df(subset_df, model_id)

    start_time = time.time()
    for idx in tqdm(range(len(generation_df))):
        if generation_df.loc[idx, "rewritten_review_text"] is not None:
            continue
        prompt = generation_df.loc[idx, "prompt"]
        try:
            generation_df.loc[idx, "rewritten_review_text"] = generate_rewrite(prompt, tokenizer, model)
        except Exception as e:
            generation_df.loc[idx, "rewritten_review_text"] = None
            generation_df.loc[idx, "generation_error"] = str(e)

        if (idx + 1) % SAVE_EVERY == 0:
            gc.collect()
            if torch.backends.mps.is_available():
                torch.mps.empty_cache()
            generation_df.to_parquet(output_path, index=False)

    generation_df.to_parquet(output_path, index=False)
    print("Done.", model_id, "| elapsed sec:", round(time.time() - start_time, 2), "->", output_path)
    return generation_df


llama_generation_df = run_generation(llama_df, LLAMA_MODEL_ID, LLAMA_OUTPUT_PATH)
qwen_generation_df = run_generation(qwen_df, QWEN_MODEL_ID, QWEN_OUTPUT_PATH)

### 생성 결과 결합

- Llama / Qwen 결과를 합쳐 셔플 후 `yelp_type3_rewritten_reviews.parquet`로 저장

In [ ]:
RANDOM_STATE = 42

llama_result_df = pd.read_parquet(LLAMA_OUTPUT_PATH)
qwen_result_df = pd.read_parquet(QWEN_OUTPUT_PATH)

combined_df = pd.concat([llama_result_df, qwen_result_df], axis=0, ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
combined_df.to_parquet(REWRITE_OUTPUT_PATH, index=False)

print("Combined rows:", len(combined_df))
print(combined_df["model_name"].value_counts())
print("Saved:", REWRITE_OUTPUT_PATH)

### Human vs AI 라벨링 20K 데이터셋 구성

각 행의 원본 리뷰(`review_text`)와 AI 재작성 리뷰(`rewritten_review_text`)를 **각각 한 행으로 펼쳐** 총 20,000행을 만든다.

| 컬럼 | 내용 |
|---|---|
| `pk` | `{review_id}_human` / `{review_id}_ai` (행 고유 복합키) |
| `review_id` | 원본 리뷰 ID (human↔ai 쌍 매칭용) |
| `text` | 리뷰 본문 (human=원본, ai=재작성) |
| `label` | `human` / `ai` |
| `source` | `human` / 모델명 |
| `review_stars`, `business_id` | 핵심 메타 |

In [ ]:
META_COLS = ["review_id", "review_stars", "business_id"]
FINAL_COLS = ["pk", "review_id", "text", "label", "source", "review_stars", "business_id"]

# human 행: 원본 리뷰
human = combined_df[META_COLS].copy()
human["text"] = combined_df["review_text"]
human["label"] = "human"
human["source"] = "human"
human["pk"] = combined_df["review_id"].astype(str) + "_human"

# ai 행: LLM 재작성 리뷰
ai = combined_df[META_COLS].copy()
ai["text"] = combined_df["rewritten_review_text"]
ai["label"] = "ai"
ai["source"] = combined_df["model_name"]
ai["pk"] = combined_df["review_id"].astype(str) + "_ai"

# 비어있는 재작성 리뷰는 ai 행만 제외 (human 행은 유지)
before = len(ai)
ai = ai[ai["text"].notna() & (ai["text"].astype(str).str.strip() != "")]
if before - len(ai):
    print(f"경고: 비어있는 rewritten_review_text {before - len(ai)}건의 ai 행 제외")

labeled_df = pd.concat([human, ai], ignore_index=True)[FINAL_COLS]

assert labeled_df["pk"].is_unique, "PK 중복 발견"
labeled_df.to_parquet(LABELED_OUTPUT_PATH, index=False)

print("Saved:", LABELED_OUTPUT_PATH, "| shape:", labeled_df.shape)
print(labeled_df["label"].value_counts())
print(labeled_df["source"].value_counts())